In [1]:
#!pip uninstall langchain langchain-core langchain-community langchain-huggingface langchain-openai chromadb -y

In [2]:
#!pip install langchain langchain-community langchain-openai langchain-huggingface chromadb sentence-transformers

In [3]:
import os
from dotenv import load_dotenv
import gradio as gr
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI


c:\Users\user\ai_projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 150
RETRIEVAL_K = 3
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "knowledge-base",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")


Loaded 3 documents


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 59 chunks


In [7]:
# Embedding Model
# ===================================
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3392.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name   # automatically persisted
)

print("Vector DB created")

Vector DB created


In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": RETRIEVAL_K}
)


In [10]:
llm = ChatOpenAI(
    model=MODEL,
    temperature=0
)


In [11]:
def build_prompt(question, context):

    return f"""
You are a helpful assistant answering questions using company documents.

Only answer using the context below.
If the answer is not in the context say: "I don't know".

Context:
{context}

Question:
{question}

Answer clearly and concisely.
"""

In [12]:
# Format Retrieved Docs
# ===================================
def format_docs(docs):

    return "\n\n".join(doc.page_content for doc in docs)

In [13]:
def ask(question):

    # Step 1: Retrieve documents
    docs = retriever.invoke(question)

    # Step 2: Combine context
    context = format_docs(docs)

    # Step 3: Build prompt
    prompt = build_prompt(question, context)

    # Step 4: Ask LLM
    response = llm.invoke(prompt)

    # Step 5: Print result
    print("\nAnswer:\n")
    print(response.content)

    print("\nSources:\n")
    for d in docs:
        print(d.metadata["source"])


In [14]:
ask("GIVE ME LINE FOR PATIENT NAME AND NATIONALITY WITH SOURCE TABLE ")


Answer:

Patient Name (English): INITCAP(p.PAT_NAME_1 || ' ' || p.PAT_NAME_2 || ' ' || p.PAT_NAME_3) || ' ' || UPPER(p.PAT_NAME_FAMILY) from table p  
Nationality: NVL(n.DESCRIPTION, 'NOT_DEFINED') from table n

Sources:

knowledge-base\SQL.txt
knowledge-base\SQL.txt
knowledge-base\SQL.txt


In [15]:
import gradio as gr

# Your ask function adapted for Gradio
def ask_ui(question):
    # Step 1: Retrieve documents
    docs = retriever.invoke(question)
    
    # Step 2: Combine context
    context = format_docs(docs)
    
    # Step 3: Build prompt
    prompt = build_prompt(question, context)
    
    # Step 4: Ask LLM
    response = llm.invoke(prompt)
    
    # Return only the answer
    return response.content

# Create Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("## 🧠 My RAG Assistant\nAsk a question and get an answer!")
    
    with gr.Row():
        question_input = gr.Textbox(label="Ask your question", placeholder="Type your question here...", lines=2)
    
    with gr.Row():
        answer_output = gr.Textbox(label="Answer", placeholder="Answer will appear here...", interactive=False, lines=10)
    
    submit_btn = gr.Button("Ask")
    
    # Connect button to function
    submit_btn.click(ask_ui, inputs=question_input, outputs=answer_output)

# Launch the app
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
